In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch, torch_scatter, numpy as np, networkx as nx
from pyvis.network import Network
from IPython.display import IFrame, display, HTML
import pandas as pd
import ipywidgets as widgets

from src.utils.graph_generation import get_dataset
from src.models.module import SALSACLRSModel, calc_metrics

from loguru import logger

# Configure loguru to suppress DEBUG messages
logger.remove()
logger.add(sys.stderr, level="INFO")

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
import os
import tkinter as tk
from tkinter import filedialog

print(f"Opening pop-up dialog to select checkpoint. Please check your taskbar or active windows...")

# We use tkinter to spawn a native OS file picker!
# This natively blocks Jupyter execution without freezing the kernel!
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)  # Ensure the popup appears in front of the browser

initial_dir = os.path.join(project_root, "data/checkpoints")
if not os.path.exists(initial_dir):
    initial_dir = project_root

CHECKPOINT_PATH = filedialog.askopenfilename(
    parent=root,
    title="Select a Checkpoint File",
    initialdir=initial_dir,
    filetypes=[("Checkpoint files", "*.ckpt"), ("All files", "*.*")]
)

# Destroy the root window to give focus back properly
root.destroy()

if not CHECKPOINT_PATH:
    raise RuntimeError("No checkpoint selected. Halting execution.")
    
print(f"✅ Successfully loaded checkpoint path: {os.path.relpath(CHECKPOINT_PATH, project_root)}")

In [ ]:

def fix_batch_attributes(batch):
    for attr in ("inputs", "outputs", "hints"):
        val = getattr(batch, attr, None)
        if isinstance(val, list) and val and isinstance(val[0], list):
            setattr(batch, attr, val[0])
    batch.inputs  = ["pos", "s"]
    batch.outputs = ["pi"]
    batch.hints   = ["reach_h", "pi_h"]
    return batch

def pred_parent_per_node(pred_pi, edge_index, num_nodes):
    _, argmax_edge = torch_scatter.scatter_max(
        pred_pi, edge_index[0], dim=-1, dim_size=num_nodes
    )
    valid = argmax_edge < edge_index.shape[1]
    parent = torch.arange(num_nodes)
    parent[valid] = edge_index[1, argmax_edge[valid]]
    return parent

def true_parent_per_node(pi, edge_index, num_nodes):
    _, argmax_edge = torch_scatter.scatter_max(
        pi.float(), edge_index[0], dim=-1, dim_size=num_nodes
    )
    valid = argmax_edge < edge_index.shape[1]
    parent = torch.arange(num_nodes)
    parent[valid] = edge_index[1, argmax_edge[valid]]
    return parent

def build_undirected_graph(edge_index, num_nodes):
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    seen = set()
    for i in range(edge_index.shape[1]):
        u, v = int(edge_index[0, i]), int(edge_index[1, i])
        key = (min(u, v), max(u, v))
        if key not in seen:
            seen.add(key)
            G.add_edge(u, v)
    return G

def draw_comparison_graph(data, pred_pi, title=None, height="800px"):
    # Visualize predicted vs GT BFS predecessor tree using Pyvis.
    num_nodes = data.s.shape[0]
    source    = data.s.argmax().item()
    G         = build_undirected_graph(data.edge_index, num_nodes)

    pred_par  = pred_parent_per_node(pred_pi, data.edge_index, num_nodes).numpy()
    true_par  = true_parent_per_node(data.pi,  data.edge_index, num_nodes).numpy()

    # Precompute layout
    pos = nx.spring_layout(G, seed=42, k=2.0 / np.sqrt(num_nodes))

    reachable   = [n for n in G.nodes() if n != source and true_par[n] != n]
    unreachable = [n for n in G.nodes() if n != source and true_par[n] == n]

    net = Network(notebook=True, width="100%", height=height, directed=True, cdn_resources='in_line')
    
    # Add nodes
    for n in G.nodes():
        if n == source:
            color = "#f4a261"
            shape = "star"
            size = 25
            label = f"{n} (Source)"
        elif n in unreachable:
            color = "#bbbbbb"
            shape = "dot"
            size = 10
            label = str(n)
        else:
            color = "#457b9d"
            shape = "dot"
            size = 15
            label = str(n)
            
        x, y = pos[n]
        net.add_node(int(n), label=label, color=color, shape=shape, size=size, x=float(x)*1000, y=float(y)*1000)
        
    # Add base edges
    for u, v in G.edges():
        net.add_edge(int(u), int(v), color="#d0d0d0", width=1, arrows="")

    # Add predicted/true edges
    for n in range(num_nodes):
        if n == source:
            continue
        pp = pred_par[n]
        tp = true_par[n]
        if pp != n:
            if pp == tp:
                net.add_edge(int(n), int(pp), color="#2a9d8f", width=3, arrows="to") # correct
            else:
                net.add_edge(int(n), int(pp), color="#e63946", width=3, arrows="to") # incorrect
        if tp != n and pp != tp:
            net.add_edge(int(n), int(tp), color="#6c757d", width=2, arrows="to", dashes=True) # missed

    # Turn off physics to keep the layout stable and zoomable
    net.toggle_physics(False)
    
    # Generate HTML
    filename = "bfs_visualization.html"
    net.show(filename)
    
    if title:
        display(HTML(f"<h3>{title}</h3>"))
    with open(filename, "r") as f:
        html_content = f.read()
    escaped_html = html_content.replace("\"", "&quot;")
    display(HTML(f'<iframe srcdoc="{escaped_html}" width="100%" height="{height}" style="border:none;"></iframe>'))

In [ ]:
# Load model directly from checkpoint. It remembers its own specs and config!
model = SALSACLRSModel.load_from_checkpoint(CHECKPOINT_PATH, map_location="cpu")
cfg = model.cfg
cfg.DATA.ROOT = os.path.join(project_root, "data")

model.eval()

# Now we can just load the test datasets using its internal configuration
test_datasets = get_dataset("test", cfg)
print(f"Loaded {len(test_datasets)} test datasets according to its config: {cfg.RUN_NAME}.")

In [ ]:
from torch_geometric.data import Batch

results = []
for name, ds in test_datasets.items():
    ga, na = [], []
    with torch.no_grad():
        for i in range(min(len(ds), 20)):
            batch = Batch.from_data_list([ds[i]])
            batch = fix_batch_attributes(batch)
            output, hints, hidden = model(batch)
            m = calc_metrics("pi", output, batch, model.specs["pi"][2])
            ga.append(m["graph_result"].float().mean().item())
            na.append(m["node_accuracy"].float().mean().item())
    results.append({"Dataset": name,
                    "Graph Accuracy": np.mean(ga),
                    "Node Accuracy":  np.mean(na)})

display(pd.DataFrame(results))


In [ ]:

from torch_geometric.data import Batch

# Choose ONE graph type to visualize explicitly instead of all of them
target_set = "ws_800"

if target_set in test_datasets:
    ds = test_datasets[target_set]
    sample = ds[0]
    batch = Batch.from_data_list([sample])
    batch = fix_batch_attributes(batch)

    with torch.no_grad():
        output, hints, hidden = model(batch)

    draw_comparison_graph(
        sample, output["pi"],
        title=f"{target_set} (N={sample.num_nodes})"
    )
else:
    print(f"Dataset {target_set} not found in test_datasets.")
